In [1]:
!pip install google-adk google-genai -q

In [2]:
!pip install litellm fastapi uvicorn httpx pydantic openai streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.1
    Uninstalling importlib_metadata-9.0.1:
      Successfully uninstalled importlib_metadata-9.0.1


In [27]:
from google.adk.agents import Agent
#from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner, InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import logging
import json
import re

print('bibliotecas importadas')

bibliotecas importadas


In [28]:
import os
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('KEY_DISCENTE') # Corrected variable name to OPENAI_API_KEY
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY # Set as environment variable
print('chave api configurada com sucesso')

chave api configurada com sucesso


In [29]:
flight_agent = Agent(
    name="flight_agent",
    model="gemini-3.6-flash",
    description="Suggests flight options for a destination.",
    instruction=(
        "Given a destination, travel dates, and budget, suggest 1-2 realistic flight options. "
        "Include airline name, price, and departure time. Ensure flights fit within the budget."
    )
)

print('configurando o agente de voo')

configurando o agente de voo


In [30]:
stay_agent = Agent(
    name="stay_agent",
    model="gemini-3.6-flash",
    description="Suggests hotel or stay options for a destination.",
    instruction=(
        "Given a destination, travel dates, and budget, suggest 2-3 hotel or stay options. "
        "Include hotel name, price per night, and location. Ensure suggestions are within budget."
    )
)

print('configurando o agente de hospedagem')

configurando o agente de hospedagem


In [31]:
activities_agent = Agent(
    name="activities_agent",
    model="gemini-3.6-flash",
    description="Suggests interesting activities for the user at a destination.",
    instruction=(
        "Given a destination, dates, and budget, suggest 2-3 engaging tourist or cultural activities. "
        "For each activity, provide a name, a short description, price estimate, and duration in hours. "
        "Respond in plain English. Keep it concise and well-formatted."
    )
)

print('configurando o agente de atividades turísticas')

configurando o agente de atividades turísticas


In [32]:

host_agent = Agent(
    name="host_agent",
    model="gemini-3.6-flash",
    description="Coordinates travel planning by calling flight, stay, and activity agents.",
    instruction=("You are a travel coordinator. Delegate tasks to the specialists:\n"
                "- flight_agent for flights\n"
                "- stay_agent for accommodation\n"
                "- activities_agent for activities\n\n"
                "Delegate to ALL three before compiling the final response.\n"
                "Then, synthesize everything into a complete itinerary."
    ),
    sub_agents=[flight_agent, stay_agent, activities_agent],
)

session_service = InMemorySessionService()
runner = Runner(
    agent=host_agent,
    app_name="host_app",
    session_service=session_service
)
USER_ID = "user_host"
SESSION_ID = "session_host"

await session_service.create_session(
    app_name="host_app",
    user_id=USER_ID,
    session_id=SESSION_ID
)

Session(id='session_host', app_name='host_app', user_id='user_host', state={}, events=[], last_update_time=1789482208.8973048)

In [33]:
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)
logging.getLogger("google_adk").setLevel(logging.CRITICAL)

async def execute(request):
    prompt = (
        f"User is flying to {request['destination']} from {request['start_date']} to {request['end_date']}, "
        f"with a budget of {request['budget']}. Suggest 2-3 activities, each with name, description, price estimate, and duration. "
        f"Respond in JSON format using the key 'activities' with a list of activity objects."
    )
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=message):
        if event.is_final_response():
            response_text = event.content.parts[0].text

            # Limpa cercas markdown (```json ... ```)
            response_text = re.sub(r"^```(?:json)?\s*", "", response_text)
            response_text = re.sub(r"\s*```$", "", response_text)
            response_text = response_text.strip()

            try:
                parsed = json.loads(response_text)
                if "activities" in parsed and isinstance(parsed["activities"], list):
                    return {"activities": parsed["activities"]}
                else:
                    print("'activities' key missing or not a list in response JSON")
                    return {"activities": response_text}  # fallback to raw text
            except json.JSONDecodeError as e:
                print("JSON parsing failed:", e)
                print("Response content:", response_text)
                return {"activities": response_text}  # fallback to raw text

In [34]:
requestTravel = {
    "destination": "Rio de Janeiro",
    "start_date": "2026-12-10",
    "end_date": "2026-12-15",
    "budget": "R$ 5.000"
}

await execute(requestTravel)

{'activities': [{'name': 'Christ the Redeemer & Corcovado Train',
   'description': 'Take a scenic rack railway train through the Tijuca Rainforest to visit the world-famous Christ the Redeemer statue atop Corcovado Mountain.',
   'price_estimate': 'R$ 130',
   'duration_hours': 3},
  {'name': 'Sugarloaf Mountain Cable Car',
   'description': 'Ride the iconic two-stage cable car to the peak of Sugarloaf Mountain for spectacular 360-degree views of Rio de Janeiro, Guanabara Bay, and Copacabana Beach.',
   'price_estimate': 'R$ 160',
   'duration_hours': 3.5},
  {'name': 'Guanabara Bay Sunset Sailing Tour',
   'description': "Enjoy a relaxing evening cruise along Guanabara Bay, taking in views of Rio's skyline, Sugarloaf Mountain, and the Niterói Contemporary Art Museum at sunset.",
   'price_estimate': 'R$ 220',
   'duration_hours': 3}]}